# Blockchain Case Study — Executive Decision

> **One notebook. All dimensions. One recommendation.**

This notebook brings together the key outputs from Notebooks 02–06 into a single executive view.
Instead of switching between five notebooks, read this one page and get the answer.

---

### Who this notebook is for

| Role | What you get |
|---|---|
| **Product Owner** | One-page decision: build it or not, in which order, at what risk |
| **Portfolio Owner** | Combined view of value, cost, risk, and deployment across all features |
| **Board / Steering** | Traffic-light recommendation backed by five quantitative dimensions |
| **Risk Manager** | Cross-dimensional risk check — no blind spots |

### Five dimensions, one decision

| # | Dimension | Source | Question answered |
|:-:|-----------|--------|-------------------|
| 1 | Business Value | Notebook 02 | How much can each feature deliver? |
| 2 | Financial Return | Notebook 03 | Is the investment worthwhile? (NPV, IRR) |
| 3 | Portfolio Optimization | Notebook 04 | Which features should we build at this budget? |
| 4 | Risk Profile | Notebook 05 | What can go wrong and how much value is at risk? |
| 5 | Deployment Cost Risk | Notebook 06 | Will deployment stay within budget if sprints run late? |

> For deep dives, follow the links to each source notebook.

---

*Previous: [06 — Deployment Risk](06-blockchain-case-study-development-risk.ipynb) · Deep dives: [02](02-blockchain-case-study.ipynb) · [03](03-blockchain-case-study-capital-budgeting.ipynb) · [04](04-blockchain-case-study-advisor.ipynb) · [05](05-blockchain-case-study-risk.ipynb) · [06](06-blockchain-case-study-development-risk.ipynb)*

## 1) Setup

Load the blockchain scenario and initialise all services. No changes needed — just run the notebook.

In [ ]:
import math

from fhs.application import (
    AdvancedPortfolioService,
    BlockchainCaseStudyService,
    BoardRecommendationService,
)
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import COLORS, show
from fhs.presentation.notebook.charts import (
    plot_delivery_market_resilience,
)

setup = notebook_setup("blockchain")
scenario = setup.scenario
risk_model = setup.risk_model
delivery_config = setup.delivery_config

if scenario is None or risk_model is None or delivery_config is None:
    raise RuntimeError("Scenario setup could not be initialized")

features = sorted(scenario.features, key=lambda x: x.name)
feature_names = [f.name for f in features]
feature_keys = sorted(scenario.features_by_key)

# Services
case = BlockchainCaseStudyService(seed=scenario.seed, scenarios=scenario.scenarios)
service = AdvancedPortfolioService.from_scenario(
    scenario, seed=scenario.seed, scenarios=scenario.scenarios
)
rec_service = BoardRecommendationService(
    seed=scenario.seed, scenarios=scenario.scenarios
)

# Core simulations (run once, reused throughout)
year1 = case.simulate_year1(scenario.features_by_key)
_ = service.simulation_results

icons = {"H1": "\U0001f4f1", "H2": "\U0001f517", "H3": "⚠️"}
colors = {"H1": COLORS.primary, "H2": COLORS.secondary, "H3": COLORS.accent}

show.info(
    f"Configuration: <code>{scenario.config_path}</code><br>"
    f"Budget: <b>EUR {scenario.budget:,.0f}</b> · "
    f"Discount rate: <b>{scenario.discount_rate:.0%}</b><br>"
    f"Features: <b>{len(features)}</b> · "
    f"Scenarios: <b>{scenario.scenarios:,}</b>"
)

---

## 2) Business Value — What can each feature deliver?

*Source: [Notebook 02 — Blockchain Case Study](02-blockchain-case-study.ipynb)*

Each feature was simulated across the configured number of Monte Carlo scenarios. The cards below show the expected business value, the downside floor (VaR 95%), and the worst-case average (CVaR 95%).

In [ ]:
cards = []
for key in feature_keys:
    expected = float(year1[key].expected_eur)
    spread = float(year1[key].std_eur) / expected if expected else 0.0
    cards.append(
        show.feature_risk_card(
            year1[key].feature.name,
            expected,
            float(year1[key].var_95_eur),
            spread,
            cvar95_eur=float(year1[key].cvar_95_eur),
            icon=icons.get(key, "\U0001f4ca"),
            color=colors.get(key, COLORS.primary),
        )
    )
show.scorecard(cards, title="Feature Business Value — All Hypotheses")

In [ ]:
portfolio_overview = case.portfolio_year1(year1)

show.year1_overview(
    case.year1_overview_rows(year1, portfolio_overview),
    title="Year 1 Overview — Business Value vs. Costs",
)

**Reading the table:** Green net values mean the feature covers its costs in Year 1. Red values are normal for multi-year programmes — see the financial analysis below.

> **Deep dive:** [Notebook 02](02-blockchain-case-study.ipynb) for risk comparison charts, CVaR analysis, and operating cost simulation.

---

## 3) Financial Return — Is the investment worthwhile?

*Source: [Notebook 03 — Capital Budgeting](03-blockchain-case-study-capital-budgeting.ipynb)*

Two financing options compared over 3 years:
- **Option A (Upfront):** Full development cost paid at Year 0
- **Option B (Installment):** Cost spread over annual payments

| Metric | What it tells you |
|--------|-------------------|
| **NPV** | Net Present Value — positive = value creation |
| **IRR** | Internal Rate of Return — must exceed the hurdle rate |
| **PI** | Profitability Index — value created per euro invested |

In [ ]:
ctx = case.capital_budgeting_context(scenario, years=3)
npv_benefit = ctx.portfolio_npv_b.expected - ctx.portfolio_npv_a.expected


# IRR formatting helper
def _irr_fmt(v: float) -> str:
    if math.isnan(v):
        return "\u221e (positive from Year 1)"
    if v <= 0:
        return "\u2264 0% — no break-even"
    return f"{v:.1%}"


irr_a_color = (
    COLORS.success
    if (
        not math.isnan(ctx.portfolio_irr_a.expected)
        and ctx.portfolio_irr_a.expected >= ctx.discount_rate
    )
    or math.isnan(ctx.portfolio_irr_a.expected)
    else COLORS.danger
)
irr_b_color = (
    COLORS.success
    if (
        not math.isnan(ctx.portfolio_irr_b.expected)
        and ctx.portfolio_irr_b.expected >= ctx.discount_rate
    )
    or math.isnan(ctx.portfolio_irr_b.expected)
    else COLORS.danger
)

show.executive(
    "Financial Investment Summary (3-Year Horizon)",
    [
        ("Total Investment", f"\u20ac{ctx.total_cost:,.0f}", COLORS.neutral),
        ("Discount Rate (hurdle)", f"{ctx.discount_rate:.0%}", COLORS.neutral),
        (
            "NPV — Option A (Upfront)",
            f"\u20ac{ctx.portfolio_npv_a.expected:,.0f}",
            COLORS.secondary,
        ),
        (
            "NPV — Option B (Installment)",
            f"\u20ac{ctx.portfolio_npv_b.expected:,.0f}",
            COLORS.secondary,
        ),
        ("NPV Advantage of Option B", f"+\u20ac{npv_benefit:,.0f}", COLORS.success),
        ("IRR — Option A", _irr_fmt(ctx.portfolio_irr_a.expected), irr_a_color),
        ("IRR — Option B", _irr_fmt(ctx.portfolio_irr_b.expected), irr_b_color),
        ("PI — Option A", f"{ctx.pi_a:.2f}\u00d7", COLORS.secondary),
        ("PI — Option B", f"{ctx.pi_b:.2f}\u00d7", COLORS.success),
    ],
    footer=f"Monte Carlo: {ctx.scenarios:,} scenarios per feature. PI = NPV / PV(investment).",
)

In [ ]:
installment_schedule = case.installment_schedule(ctx.features_by_key, years=3)
nonzero_installments = [amount for amount in installment_schedule if amount > 0]
if len({round(amount, 2) for amount in nonzero_installments}) == 1:
    option_b_label = (
        f"Option B \u20ac{nonzero_installments[0]:,.0f}/yr over "
        f"{len(nonzero_installments)} years"
    )
else:
    option_b_label = (
        "Option B mixed schedule ("
        + " \u2192 ".join(f"\u20ac{amount:,.0f}" for amount in nonzero_installments)
        + ")"
    )

rec_fin = case.financing_recommendation(
    npv_a=ctx.portfolio_npv_a,
    npv_b=ctx.portfolio_npv_b,
    irr_a=ctx.portfolio_irr_a,
    irr_b=ctx.portfolio_irr_b,
    total_investment=ctx.total_cost,
    total_annual_installment=ctx.total_annual_installment,
    installment_years=ctx.total_installment_years,
    discount_rate=ctx.discount_rate,
    installment_schedule=installment_schedule,
)
show.financing_recommendation(
    rec_fin,
    title=(
        f"Financing Decision — "
        f"Option A \u20ac{ctx.total_cost:,.0f} upfront vs. "
        f"{option_b_label}"
    ),
)

> **Deep dive:** [Notebook 03](03-blockchain-case-study-capital-budgeting.ipynb) for NPV rate curves, IRR hurdle gauges, and per-feature cash flow tables.

---

## 4) Portfolio Optimization — Which features should we build?

*Source: [Notebook 04 — Portfolio Advisor](04-blockchain-case-study-advisor.ipynb)*

The budget optimizer selects the best feature combination within the budget. We compare two strategies:
- **Business value floor:** maximise the worst-case business value (downside protection)
- **3-year NPV:** maximise the 3-year net present value (growth-oriented)

In [ ]:
ilp_result = service.optimize(
    solver="ilp", budget=scenario.budget, strategy="var_floor"
)
selected_names_list = list(ilp_result.recommended_features)
investment = float(ilp_result.total_cost)

show.optimizer(
    selected_names_list,
    investment,
    float(ilp_result.portfolio_expected),
    float(ilp_result.portfolio_var_95),
    float(scenario.budget) - investment,
    title="Optimal Portfolio (Business Value Floor strategy, 100% Budget)",
    objective="Business Value Floor — protect the worst case",
)

In [ ]:
discount_rate = scenario.discount_rate
full_budget = float(scenario.budget)
budget_levels = {
    "25% Budget": full_budget * 0.25,
    "50% Budget": full_budget * 0.50,
    "100% Budget": full_budget,
}

ilp_year1 = service.budget_sensitivity(
    budget_levels, solver="ilp", strategy="npv_year1"
)
ilp_3year = service.budget_sensitivity(
    budget_levels, solver="ilp", strategy="npv_3year"
)

decision_rows = service.decisions.npv_decision_table_rows(
    budget_levels, ilp_year1, ilp_3year, discount_rate=discount_rate
)
show.npv_decision_table(decision_rows)

**Reading the table:** Each row is a budget level. The optimizer picks different features depending on budget and horizon. When both horizons agree, confidence is high.

> **Deep dive:** [Notebook 04](04-blockchain-case-study-advisor.ipynb) for per-feature NPV comparison and decision governance rules.

---

## 5) Risk Profile — What can go wrong?

*Source: [Notebook 05 — Risk Dashboard](05-blockchain-case-study-risk.ipynb)*

Four risk dimensions are applied sequentially to the selected portfolio:

| Risk | What happens | Probability |
|------|-------------|-------------|
| **Development** | Feature not delivered — business value drops to zero | Per feature (LLP) |
| **Market** | Market shock — business value × 0.85 | 20% |
| **Component** | Shared platform fails — business value × factor | Per shared component |
| **Global** | Crisis — business value × 0.60 | 5% |

In [ ]:
portfolio_layers = service.layers.simulate_portfolio_risk_layers(
    selected_names_list, risk_model=risk_model, seed=scenario.seed
)
before_risk = portfolio_layers.base.expected
after_risk = portfolio_layers.after_risk_3.expected

# Identify dominant risk dimension
delivery_loss = before_risk - portfolio_layers.after_risk_1.expected
market_loss = (
    portfolio_layers.after_risk_1.expected - portfolio_layers.after_risk_2.expected
)
component_loss = (
    portfolio_layers.after_risk_2.expected - portfolio_layers.after_component.expected
)
global_loss = (
    portfolio_layers.after_component.expected - portfolio_layers.after_risk_3.expected
)

losses = {
    "Development failure": delivery_loss,
    "Market shock": market_loss,
    "Component failure": component_loss,
    "Global crisis": global_loss,
}
dominant = max(losses, key=losses.get)
dominant_pct = 100.0 * losses[dominant] / before_risk if before_risk > 0 else 0.0

show.executive_risk_summary(
    before_risk,
    after_risk,
    investment,
    f"{dominant} ({dominant_pct:.0f}% of base value)",
)

In [ ]:
show.portfolio_risk_waterfall_detail(portfolio_layers, risk_model, features, investment)

In [ ]:
# Development x Market heatmap (3x3)
llp_factors = [0.5, 1.0, 1.5]
r2_levels = [0.10, 0.20, 0.30]
heatmap_data = {}
for llp_f in llp_factors:
    for p2 in r2_levels:
        heatmap_data[(llp_f, p2)] = service.layers.simulate_portfolio_risk_layers(
            selected_names_list,
            risk_model=risk_model,
            risk1_factor=llp_f,
            risk2_probability=p2,
            seed=scenario.seed,
        ).after_risk_3.expected

_, resilience = plot_delivery_market_resilience(
    heatmap_data,
    investment=investment,
    llp_factors=tuple(llp_factors),
    market_levels=tuple(r2_levels),
)

show.metrics(
    [
        (
            "Base case (current development, Market 20%)",
            f"EUR {resilience['base_k']:,.0f}k",
            COLORS.primary,
        ),
        (
            "If development risk halved",
            f"EUR {resilience['improved_k']:,.0f}k  (+{resilience['delivery_gain_k']:,.0f}k)",
            COLORS.success,
        ),
        (
            "If development risk worsens (+50%)",
            f"EUR {resilience['worsened_k']:,.0f}k  ({-resilience['delivery_loss_stress_k']:,.0f}k)",
            COLORS.danger,
        ),
    ],
    title="Development Risk Mitigation Impact",
)

**Key insight:** Reducing development failure probability has the highest return on effort — it is both the largest risk and the one within your control.

> **Deep dive:** [Notebook 05](05-blockchain-case-study-risk.ipynb) for feature risk profiles, sensitivity tables, and ranking strategy comparison.

---

## 6) Deployment Cost Risk — Will deployment stay within budget?

*Source: [Notebook 06 — Deployment Risk](06-blockchain-case-study-development-risk.ipynb)*

Sprint overruns are simulated using a lognormal delay model. The simulation shows how much more each feature could cost when teams run late.

In [ ]:
configured_scenarios = int(scenario.scenarios)
delivery_config_run = delivery_config.model_copy(
    update={"scenarios": configured_scenarios}
)

delivery_results = service.delivery.simulate_risk(
    feature_names, delivery_config=delivery_config_run, seed=scenario.seed
)

show.delay_summary(delivery_results, feature_names=feature_names)

In [ ]:
show.delivery_cost_risk(features, delivery_results)

In [ ]:
show.budget_fit_and_select(features, delivery_results, float(scenario.budget))

> **Deep dive:** [Notebook 06](06-blockchain-case-study-development-risk.ipynb) for sprint delay distributions, capacity checks, and cost comparison charts.

---

## 7) Combined Recommendation — The Verdict

All five dimensions are now computed. This section synthesises them into a single, actionable recommendation.

In [ ]:
# ── Gather verdicts from all dimensions ──────────────────────────────

# Dimension 1: Business Value (NB 02)
selected_features = [year1[key].feature for key in feature_keys]
recommendation = rec_service.generate_recommendation(
    selected_features,
    scenario.biz_values,
    discount_rate=scenario.discount_rate,
    year1_results=year1,
)
bv_signal = recommendation.go_decision  # GO / CONDITIONAL_GO / REVIEW
bv_expected = recommendation.total_portfolio_expected
bv_floor = recommendation.total_portfolio_var95

# Dimension 2: Financial Return (NB 03)
npv_positive = ctx.portfolio_npv_a.expected > 0
irr_above_hurdle = (
    math.isnan(ctx.portfolio_irr_a.expected)
    or ctx.portfolio_irr_a.expected >= ctx.discount_rate
)
fin_signal = "GO" if npv_positive and irr_above_hurdle else "REVIEW"

# Dimension 3: Portfolio Optimization (NB 04)
opt_count = len(selected_names_list)
opt_total = len(features)
opt_signal = "GO" if opt_count == opt_total else "CONDITIONAL_GO"

# Dimension 4: Risk Profile (NB 05)
net_after_risk = after_risk - investment
retention_pct = (after_risk / before_risk * 100) if before_risk > 0 else 0.0
risk_signal = "GO" if net_after_risk > 0 else "REVIEW"

# Dimension 5: Deployment Cost Risk (NB 06)
total_planned_cost = sum(float(f.development_cost) for f in features)
delivery_cvar_values = []
for fname in feature_names:
    if fname in delivery_results:
        dr = delivery_results[fname]
        delivery_cvar_values.append(float(dr.cost_cvar))
total_delivery_cvar = sum(delivery_cvar_values)
delivery_pressure = (
    (total_delivery_cvar / total_planned_cost - 1) if total_planned_cost > 0 else 0.0
)
delivery_signal = (
    "GO"
    if delivery_pressure < 0.15
    else ("CONDITIONAL_GO" if delivery_pressure < 0.30 else "REVIEW")
)

In [ ]:
# ── Dimension comparison table ────────────────────────────────────────


def _signal_badge(signal: str) -> str:
    color_map = {
        "GO": COLORS.success,
        "CONDITIONAL_GO": COLORS.warning,
        "REVIEW": COLORS.danger,
    }
    label_map = {
        "GO": "GO",
        "CONDITIONAL_GO": "CONDITIONAL",
        "REVIEW": "REVIEW",
    }
    c = color_map.get(signal, COLORS.neutral)
    return (
        f'<span style="display:inline-block;padding:3px 10px;border-radius:4px;'
        f"background:{c};color:#fff;font-weight:700;font-size:12px;"
        f'letter-spacing:0.3px">{label_map.get(signal, signal)}</span>'
    )


dimensions = [
    (
        "Business Value",
        bv_signal,
        f"Expected: EUR {bv_expected:,.0f} \u00b7 Floor: EUR {bv_floor:,.0f}",
        "Board recommendation from Monte Carlo simulation",
    ),
    (
        "Financial Return",
        fin_signal,
        f"NPV (A): EUR {ctx.portfolio_npv_a.expected:,.0f} \u00b7 IRR (A): {_irr_fmt(ctx.portfolio_irr_a.expected)}",
        "NPV positive and IRR above hurdle"
        if fin_signal == "GO"
        else "NPV or IRR below threshold",
    ),
    (
        "Portfolio Optimization",
        opt_signal,
        f"{opt_count}/{opt_total} features selected at 100% budget",
        "All features fit the budget"
        if opt_signal == "GO"
        else f"Only {opt_count} of {opt_total} features fit",
    ),
    (
        "Risk Profile",
        risk_signal,
        f"Retention: {retention_pct:.0f}% \u00b7 Net after risk: EUR {net_after_risk:,.0f}",
        "Portfolio profitable after all risk layers"
        if risk_signal == "GO"
        else "Risk erodes profitability",
    ),
    (
        "Deployment Cost Risk",
        delivery_signal,
        f"Budget pressure: {delivery_pressure:+.0%} \u00b7 CVaR: EUR {total_delivery_cvar:,.0f}",
        "Cost overrun within tolerance"
        if delivery_signal == "GO"
        else "Deployment cost risk needs attention",
    ),
]

rows_html = "".join(
    f'<tr style="border-bottom:1px solid {COLORS.border}">'
    f'<td style="padding:10px 12px;font-weight:600;color:inherit">{dim}</td>'
    f'<td style="padding:10px 12px;text-align:center">{_signal_badge(sig)}</td>'
    f'<td style="padding:10px 12px;font-family:monospace;font-size:13px;color:inherit">{metric}</td>'
    f'<td style="padding:10px 12px;font-size:13px;color:inherit">{action}</td>'
    f"</tr>"
    for dim, sig, metric, action in dimensions
)

show(
    f'<div style="border:1px solid {COLORS.border};border-radius:8px;'
    f'padding:20px;margin:12px 0;background:transparent;color:inherit">'
    f'<h3 style="margin:0 0 14px 0;color:inherit">Dimension Comparison</h3>'
    f'<table style="width:100%;border-collapse:collapse">'
    f'<thead><tr style="border-bottom:2px solid {COLORS.border}">'
    f'<th style="padding:8px 12px;text-align:left;color:inherit">Dimension</th>'
    f'<th style="padding:8px 12px;text-align:center;color:inherit">Signal</th>'
    f'<th style="padding:8px 12px;text-align:left;color:inherit">Key Metric</th>'
    f'<th style="padding:8px 12px;text-align:left;color:inherit">Assessment</th>'
    f"</tr></thead>"
    f"<tbody>{rows_html}</tbody></table></div>"
)

In [ ]:
# ── Overall recommendation ────────────────────────────────────────────

signals = [bv_signal, fin_signal, opt_signal, risk_signal, delivery_signal]
go_count = signals.count("GO")
review_count = signals.count("REVIEW")

if review_count >= 2:
    overall = "REVIEW"
    overall_color = COLORS.danger
    overall_icon = "\u26d4"
    overall_label = "REVIEW REQUIRED"
elif go_count == 5:
    overall = "GO"
    overall_color = COLORS.success
    overall_icon = "\u2705"
    overall_label = "GO"
else:
    overall = "CONDITIONAL_GO"
    overall_color = COLORS.warning
    overall_icon = "\u26a0\ufe0f"
    overall_label = "CONDITIONAL GO"

# Build reasoning
go_dims = [
    d for d, s in zip([x[0] for x in dimensions], signals, strict=False) if s == "GO"
]
issue_dims = [
    d for d, s in zip([x[0] for x in dimensions], signals, strict=False) if s != "GO"
]

reasoning_parts = []
if go_dims:
    reasoning_parts.append(f"<b>Strengths:</b> {', '.join(go_dims)} — all clear.")
if issue_dims:
    reasoning_parts.append(
        f"<b>Attention needed:</b> {', '.join(issue_dims)} — see dimension table above."
    )
reasoning_html = "<br>".join(reasoning_parts)

show(
    f'<div style="border:2px solid {overall_color};border-radius:10px;'
    f'padding:24px 28px;margin:16px 0;background:transparent;color:inherit">'
    f'<div style="display:flex;align-items:center;gap:14px;margin-bottom:14px">'
    f'<span style="font-size:36px">{overall_icon}</span>'
    f"<div>"
    f'<div style="font-size:11px;text-transform:uppercase;letter-spacing:0.5px;'
    f'font-weight:700;color:{overall_color}">Overall Recommendation</div>'
    f'<div style="font-size:28px;font-weight:800;color:{overall_color};'
    f'line-height:1.1">{overall_label}</div>'
    f"</div></div>"
    f'<div style="font-size:14px;line-height:1.6;color:inherit;'
    f'border-top:1px solid {COLORS.border};padding-top:12px">'
    f"{reasoning_html}</div>"
    f'<div style="margin-top:12px;padding:10px 14px;border-left:4px solid {overall_color};'
    f'background:transparent;font-size:13px;line-height:1.5;color:inherit">'
    f"<b>{go_count}/5</b> dimensions show GO \u00b7 "
    f"<b>{signals.count('CONDITIONAL_GO')}/5</b> conditional \u00b7 "
    f"<b>{review_count}/5</b> need review"
    f"</div></div>"
)

### What-If Quick Check — How does the picture change?

Three scenarios testing how robust the recommendation is under different conditions.

In [ ]:
# ── Scenario comparison: optimistic / base / stressed ─────────────────

scenario_configs = [
    ("Optimistic", 0.5, 0.10),  # halved development risk, 10% market
    ("Base case", 1.0, 0.20),  # current assumptions
    ("Stressed", 1.5, 0.30),  # 50% worse development, 30% market
]

scenario_rows = []
for label, llp_f, mkt_p in scenario_configs:
    result = service.layers.simulate_portfolio_risk_layers(
        selected_names_list,
        risk_model=risk_model,
        risk1_factor=llp_f,
        risk2_probability=mkt_p,
        seed=scenario.seed,
    )
    after = result.after_risk_3.expected
    net = after - investment
    profitable = net > 0
    scenario_rows.append((label, llp_f, mkt_p, after, net, profitable))

rows_html = "".join(
    f'<tr style="border-bottom:1px solid {COLORS.border}'
    f'{(";font-weight:600" if label == "Base case" else "")}">'
    f'<td style="padding:10px 12px;color:inherit">{label}</td>'
    f'<td style="padding:10px 12px;text-align:center;color:inherit">{llp_f:.1f}\u00d7</td>'
    f'<td style="padding:10px 12px;text-align:center;color:inherit">{mkt_p:.0%}</td>'
    f'<td style="padding:10px 12px;text-align:right;font-family:monospace;color:inherit">EUR {after:,.0f}</td>'
    f'<td style="padding:10px 12px;text-align:right;font-family:monospace;'
    f'color:{COLORS.success if profitable else COLORS.danger}">'
    f"{'' if net < 0 else '+'}EUR {net:,.0f}</td>"
    f'<td style="padding:10px 12px;text-align:center">'
    f"{_signal_badge('GO' if profitable else 'REVIEW')}</td>"
    f"</tr>"
    for label, llp_f, mkt_p, after, net, profitable in scenario_rows
)

show(
    f'<div style="border:1px solid {COLORS.border};border-radius:8px;'
    f'padding:20px;margin:12px 0;background:transparent;color:inherit">'
    f'<h3 style="margin:0 0 14px 0;color:inherit">What-If Scenarios — Portfolio Profitability</h3>'
    f'<table style="width:100%;border-collapse:collapse">'
    f'<thead><tr style="border-bottom:2px solid {COLORS.border}">'
    f'<th style="padding:8px 12px;text-align:left;color:inherit">Scenario</th>'
    f'<th style="padding:8px 12px;text-align:center;color:inherit">Development</th>'
    f'<th style="padding:8px 12px;text-align:center;color:inherit">Market</th>'
    f'<th style="padding:8px 12px;text-align:right;color:inherit">BV after Risk</th>'
    f'<th style="padding:8px 12px;text-align:right;color:inherit">Net Profit</th>'
    f'<th style="padding:8px 12px;text-align:center;color:inherit">Signal</th>'
    f"</tr></thead>"
    f"<tbody>{rows_html}</tbody></table>"
    f'<div style="margin-top:10px;font-size:12px;color:inherit">'
    f"Investment: EUR {investment:,.0f} \u00b7 "
    f"Development: LLP factor applied to all features \u00b7 "
    f"Market: shock probability (Component + Global at base rates)"
    f"</div></div>"
)

---

## What To Do Next

### If the recommendation is GO

1. **Confirm financing option** — use the installment view (Option B) to reduce Year-1 cash outflow
2. **Set development milestones** — use Notebook 06 sprint plan as baseline for review gates
3. **Monitor risk** — rerun this notebook quarterly with updated development probabilities

### If the recommendation is CONDITIONAL GO

1. **Check which dimensions need attention** — see the dimension table above
2. **Reduce development risk first** — it is usually the biggest lever (see heatmap in Section 5)
3. **Consider phasing** — start with the features that pass all checks, add others later

### If the recommendation is REVIEW

1. **Do not proceed without mitigation** — two or more dimensions show risk
2. **Prioritise the red dimensions** — fix the highest-impact issue first
3. **Rerun after changes** — adjust `blockchain.yaml` and rerun this notebook to see if the recommendation improves

---

## Board Summary — Phase Rollout

In [ ]:
phase_colors = [COLORS.secondary, COLORS.secondary, COLORS.accent]
show.kpi(
    *(
        show.kpi_card(value, label, color=phase_colors[min(i, len(phase_colors) - 1)])
        for i, (value, label) in enumerate(recommendation.kpi_phase_data)
    )
)
show.metrics(recommendation.portfolio_kpi_rows, title="Portfolio — Combined View")

if recommendation.decision_level == "success":
    show.success(recommendation.decision_message)
else:
    show.warning(recommendation.decision_message)

show.note(
    f"Based on {scenario.scenarios:,} Monte Carlo scenarios per hypothesis",
    compact=True,
)

---

## Notebook Navigation

| # | Notebook | What you learn |
|:-:|----------|----------------|
| 01 | [Getting Started](01-getting-started.ipynb) | One feature, simulation basics, business value floor dashboard |
| 02 | [Blockchain Case Study](02-blockchain-case-study.ipynb) | Feature risks, portfolio baseline, board recommendation |
| 03 | [Capital Budgeting](03-blockchain-case-study-capital-budgeting.ipynb) | NPV, IRR, upfront vs. installment financing |
| 04 | [Portfolio Advisor](04-blockchain-case-study-advisor.ipynb) | ILP optimisation, budget sensitivity |
| 05 | [Risk Dashboard](05-blockchain-case-study-risk.ipynb) | Risk layers, sensitivity, stress scenarios |
| 06 | [Deployment Risk](06-blockchain-case-study-development-risk.ipynb) | Sprint overruns, cost simulation, budget fit |
| **07** | **Executive Decision** | **\u2190 You are here** |
| A01 | [Portfolio Advisor](advanced/01-portfolio-advisor.ipynb) | Solver comparison, feature ranking, HHI |
| A02 | [Portfolio Risk Dashboard](advanced/02-portfolio-risk-dashboard.ipynb) | Risk layers, stress scenarios, budget risk path |

---

### Quick Reference

| Term | Definition |
|------|------------|
| **Business Value Floor 95** | 5th percentile of Monte Carlo simulation. 95% of scenarios exceed this floor. |
| **CVaR 95%** | Average of the worst 5% of scenarios. More conservative than VaR alone. |
| **NPV** | Net Present Value — sum of discounted cash flows minus investment. Positive = value creation. |
| **IRR** | Internal Rate of Return — discount rate where NPV = 0. Must exceed hurdle rate. |
| **Retention rate** | Percentage of base business value surviving all risk layers. |
| **Budget pressure** | (CVaR 95% / planned cost) \u2212 1. How far worst-case cost exceeds plan. |
| **LLP** | Likelihood of non-completion — probability a feature never reaches production. |

---

**Previous:** [NB 06: Deployment Risk](06-blockchain-case-study-development-risk.ipynb)